# LangChain Agent 스켈레톤 구현(Gemma)  
## Gemma-3-12b 모델을 LLM을 사용한 ReAct Agent를 만들고,   
## ReAct Agent의 동작을 확인하자   

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
import torch; torch._dynamo.config.recompile_limit = 64;

In [2]:
import os
from google.colab import userdata

# Colab Secrets에서 토큰 읽어오기
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
!hf auth whoami

user:  gshong
orgs:  LLM2506,aicmap


## LangChain lib import

In [3]:
!pip install langchain==0.3.27 langchain-core==0.3.79  -q

!pip show langchain langchain-core huggingface_hub | grep -E "Name: |Version:"
## 2511
# Name: langchain
# Version: 0.3.27
# Name: langchain-core
# Version: 0.3.79
# Name: huggingface-hub
# Version: 0.36.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.8/449.8 kB 45.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.0.6 requires langchain-core>=1.0.0, but you have langchain-core 0.3.79 which is incompatible.
Name: langchain
Version: 0.3.27
Name: langchain-core
Version: 0.3.79
Name: huggingface-hub
Version: 0.36.0


In [4]:
#!pip install langchain==0.3.27 langchain-core==0.3.79  -q
!pip install --prefer-binary langchain-community==0.3.30  -q
# google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.1.4 requires msgspec, which is not installed.
unsloth-zoo 2026.1.4 requires tyro, which is not installed.
unsloth 2026.1.4 requires tyro, which is not installed.
unsloth-zoo 2026.1.4 requires torchao>=0.13.0, but you have torchao 0.10.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [5]:
os.environ['LANGCHAIN_DEBUG'] = '0' # Set LANGCHAIN_DEBUG to '0' to bypass the debug check

### llama-cpp 설치

In [6]:
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 551.6/551.6 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.8 MB/s eta 0:00:00


## LLM 래퍼 설정  
> Wrapper는 LLM의 종류와 특성을 LangChain에 맞도록 숨겨주는 역할을 함, Interface 통일, LLM 교체를 쉽게 할 수 있음     
> OpenAI의 GPT-4 모델을 사용합니다. temperature=0으로 설정하여 일관성 있는 답변을 유도합니다.

https://huggingface.co/unsloth/collections    
https://huggingface.co/collections/unsloth/gemma-3    
https://huggingface.co/unsloth/gemma-3-12b-it-qat-GGUF   


In [7]:
%%time
# LlamaCpp로 모델 로드 : GGUF 모델 다운로드
from huggingface_hub import hf_hub_download
from langchain_community.llms import LlamaCpp

model_repo = "unsloth/gemma-3-12b-it-qat-GGUF" # 23.5(FP16) GB
model_file = "gemma-3-12b-it-qat-Q4_K_M.gguf"  # 7.3 GB

model_path = hf_hub_download(repo_id=model_repo, filename=model_file, local_dir="./")

llm = LlamaCpp(
    model_path=model_path,
    n_gpu_layers=40,  # GPU에 최대 레이어 오프로드 (8B 모델에 적합)
    n_batch=512,
    n_ctx=3096,    # 컨텍스트 길이
    f16_kv=True,   # FP16 메모리 최적화
    verbose=False, #True,
    temperature=0.3,
    max_tokens=256
)
# Wall time: 46.8 s

gemma-3-12b-it-qat-Q4_K_M.gguf:   0%|          | 0.00/7.30G [00:00<?, ?B/s]

llama_context: n_ctx_per_seq (3096) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
llama_kv_cache_unified_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


CPU times: user 14.6 s, sys: 21.8 s, total: 36.4 s
Wall time: 28.9 s


In [ ]:
%%time

query = "반도체 공정에서 웨이퍼란 무엇인가요?"
response = llm.invoke(query)#, max_tokens=512)
print("Query:", query)
print("Response:", response)
# Wall time: 21.8 s

Query: 반도체 공정에서 웨이퍼란 무엇인가요?
Response: 

웨이퍼는 반도체 칩을 만드는 데 사용되는 기본 재료입니다. 일반적으로 실리콘으로 만들어지며, 매우 얇고 원형의 판 모양입니다. 웨이퍼 위에 수많은 반도체 칩(집적 회로)이 제작됩니다.

**좀 더 자세히 설명하면:**

*   **재질:** 대부분 실리콘(Si)을 사용하지만, 갈륨비소(GaAs), 실리콘 카바이드(SiC) 등 다른 재료도 특정 용도로 사용됩니다.
*   **크기 (직경):** 현재 상용되는 웨이퍼는 주로 300mm(12인치) 크기를 사용하며, 더 큰 450mm 웨이퍼로의 전환을 위한 연구가 진행 중입니다. 작은 직경의 웨이퍼(예: 6인치, 8인치)도 여전히 특정 용도로 사용됩니다.
*   **두께:** 웨이퍼는 매우 얇으며, 일반적으로 수백 마이크로미터 정도 두께를 가집니다.
*   **제조 과정:** 고순도 실리콘을 녹여 단결정을 성장시키
CPU times: user 1min 50s, sys: 260 ms, total: 1min 51s
Wall time: 23.2 s


In [ ]:
%%time
## llm.invoke 대신 llm.stream 사용
for token in llm.stream(query):
    print(token, end="", flush=True)



웨이퍼는 반도체 칩을 만드는 데 사용되는 기본 재료입니다. 일반적으로 실리콘으로 만들어지며, 원반 모양의 얇은 조각입니다. 웨이퍼 위에 수많은 반도체 칩(집적 회로 또는 IC)이 제조됩니다.

다음은 웨이퍼에 대한 자세한 내용입니다.

*   **재료:** 웨이퍼는 일반적으로 실리콘으로 만들어지지만, 갈륨 아연석이나 규소 카바이드와 같은 다른 재료도 사용될 수 있습니다.
*   **크기:** 웨이퍼 크기는 200mm(8인치)에서 330mm(12인치)까지 다양하며, 더 큰 크기의 웨이퍼가 점점 더 일반화되고 있습니다. 웨이퍼의 크기가 클수록 하나의 웨이퍼에서 생산되는 반도체 칩의 수가 많아지므로 생산 비용이 절감됩니다.
*   **두께:** 웨이퍼는 매우 얇으며 일반적으로 0.5mm에서 1mm 사이입니다.
*   **제조 공정:** 웨이퍼는 실리콘 원료를 녹여 단결정을CPU times: user 1min 45s, sys: 223 ms, total: 1min 46s
Wall time: 22.6 s


In [8]:
import torch
# 주요 라이브러리 임포트
from langchain.prompts import PromptTemplate
from langchain.agents import initialize_agent, Tool, AgentType
from langchain_core.messages import SystemMessage

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

## 매뉴얼 파일 생성


In [9]:
# 매뉴얼 파일 생성
manual_content = """
= DUV (Deep Ultraviolet) 공정 매뉴얼 =

문서 버전: 1.2
작성일: 2025-06-12

== 1. 개요 ==
DUV 리소그래피는 248nm 또는 193nm 파장의 빛을 사용하여 웨이퍼에 미세 회로 패턴을 형성하는 핵심 공정입니다.

== 2. 주요 단계 ==
2.1 웨이퍼 준비 (Wafer Preparation)
- 웨이퍼 세척 및 표면 처리

2.2 감광액 도포 (PR Coating)
- 스핀 코팅 방식을 사용하여 균일한 두께의 감광액(PR) 막을 형성합니다.

2.3 노광 (Exposure)
- 마스크에 설계된 패턴을 빛을 이용해 웨이퍼의 감광액 위로 전사시킵니다.
- **핵심 파라미터:** 노광 에너지(Dose), 초점(Focus)이 패턴의 정밀도를 결정합니다.
- 노광 장비의 정렬(Align) 정확도가 수율에 큰 영향을 미칩니다.

2.4 현상 (Development)
- 노광된 영역 또는 노광되지 않은 영역의 감광액을 선택적으로 제거하여 패턴을 완성합니다.
"""

with open("DUV_manual.txt", "w") as f:
    f.write(manual_content)

print("'DUV_manual.txt' 매뉴얼 파일 생성.")

def load_manual(path: str) -> str:
    """지정된 경로(path)의 텍스트 파일을 읽어 그 내용을 문자열로 반환합니다."""
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError:
        return f"오류: '{path}' 파일을 찾을 수 없습니다."

'DUV_manual.txt' 매뉴얼 파일 생성.


## 반도체 데이터베이스 정의


In [10]:
# Tool 구현에 필요한 모든 데이터와 함수 정의

# 반도체 데이터베이스 정의: 'aliases' 키를 추가하여 영어 이름 포함
semiconductor_processes = {
    "포토리소그래피": {
        "aliases": ["photolithography", "photo"],
        "주요파라미터": "노광량(Exposure Dose), 초점(Focus), 감광액(PR) 두께",
        "설명": "반도체 웨이퍼 위에 회로 패턴을 형성하는 핵심 공정",
        "단위": "노광량(mJ/cm²), 초점(μm), PR두께(nm)",
        "중요도": "패턴 해상도와 정밀도를 결정하는 가장 중요한 공정"
    },
    "식각": {
        "aliases": ["etching", "etch"],
        "주요파라미터": "가스 유량, 압력(Pressure), RF 파워, 식각 시간",
        "설명": "불필요한 물질을 제거하여 원하는 패턴을 만드는 공정",
        "단위": "가스유량(sccm), 압력(mTorr), RF파워(W), 시간(min)",
        "중요도": "선폭 제어와 프로파일 형성에 핵심적"
    },
    "증착": {
        "aliases": ["deposition", "cvd", "pvd"],
        "주요파라미터": "온도, 압력, 전구체(Precursor) 유량",
        "설명": "웨이퍼 위에 얇은 막을 형성하는 공정",
        "단위": "온도(°C), 압력(Torr), 유량(sccm)",
        "중요도": "막질과 두께 균일성을 결정"
    }
}
# Tool로 사용할 함수들 정의

# 한국어 키와 영어 별칭(aliases)을 모두 검색
def process_lookup(query: str) -> str:
    """
    사용자 질문(query)에서 공정 이름(한/영)을 찾아, 해당 공정의 주요 파라미터, 설명, 단위, 중요도 등 상세 정보를 반환합니다.
    '포토리소그래피', '식각', '증착' 및 관련 영어 이름으로 정보를 조회할 수 있습니다.
    """
    print(f"🔍 [고급 검색] '{query}' 분석 중...")
    search_term = query.lower() # 검색어를 소문자로 변환하여 비교 정확도 높임

    for process_name, info in semiconductor_processes.items():
        # 한국어 공정 이름 또는 영어 별칭(aliases) 중에 일치하는 것이 있는지 확인
        if process_name in search_term or any(alias in search_term for alias in info.get("aliases", [])):
            detailed_info = f'''
====== {process_name} 공정 상세 정보 ======
- 주요 파라미터: {info['주요파라미터']}
- 측정 단위: {info['단위']}
- 공정 설명: {info['설명']}
- 중요도: {info['중요도']}
==========================================
'''
            print(f"[검색 성공] {process_name} 공정 정보 발견")
            return detailed_info

    available = ", ".join(semiconductor_processes.keys())
    result = f"해당 공정을 찾을 수 없습니다. 사용 가능한 공정: {available}"
    print(f"[검색 실패] {result}")
    return result

## Tool 리스트 정의


In [11]:
# 2개의 Tool로 구성된 리스트 정의
tools = [
    Tool(
        name="Semiconductor Process DB Lookup",
        func=process_lookup,
        description="반도체 공정(포토리소그래피, 식각, 증착)에 대한 상세 정보(주요 파라미터, 설명, 단위, 중요도)를 찾을 때 사용합니다. 사용자 질문에 공정 이름이 포함되어 있어야 합니다."
    ),
    Tool(
        name="Manual Loader",
        func=load_manual,
        description="로컬 텍스트 파일 형식의 매뉴얼을 읽을 때 사용합니다. 'DUV_manual.txt'와 같이 정확한 파일 경로를 입력해야 합니다."
    )
]


## LangChain Agent 생성 (ReAct Agent)  
> agent: 사용할 에이전트의 유형을 지정합니다. (e.g., ZERO_SHOT_REACT_DESCRIPTION)   
 tools: 에이전트가 사용할 도구 목록  
 llm: 에이전트의 추론을 담당할 언어 모델  
 verbose=True: 에이전트의 생각(Thought)과 행동(Action) 과정을 모두 출력하여 디버깅에 용이하게 합니다.

**AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION**

>ReAct 루프를 채팅 모델로 수행 :  
 모델이 Thought → Action → Action Input → Observation ... → Final Answer의 형식으로 추론/행동을 번갈아 수행  
>툴 설명(Descriptions)을 프롬프트에 주입하고, 모델은 이 설명을 바탕으로 어떤 툴을 언제 쓸지 스스로 결정함  
> Zero-shot이라 예시 샘플(few-shot)을 기본 포함하지 않으며, 메모리는 기본 내장하지 않음

>에이전트는 내부적으로 프롬프트 템플릿을 하나 만들고, 거기에:  
사용할 수 있는 Tool 목록  
각 Tool의 이름과 설명  
“어떻게 Tool을 선택하고 호출해야 하는지”에 대한 규칙  
을 문자열로 넣어서 LLM에게 전달합니다.

In [12]:
from langchain.agents import initialize_agent, AgentType

# 시스템 프롬프트 내용 정의 : 역할과 답변형식 지시
system_message_content = "You are a helpful assistant specializing in semiconductor processes. Your primary goal is to provide accurate and detailed information to the user. All final answers must be written clearly and politely in Korean."
system_message_content = "귀하는 반도체 공정을 전문으로 하는 친절한 보조원입니다. 귀하의 주요 목표는 사용자에게 정확하고 자세한 정보를 제공하는 것입니다. 모든 최종 답변은 한국어로 명확하고 정중하게 작성해 주시기 바랍니다."
agent_kwargs = {"system_message": SystemMessage(content=system_message_content)}

# LangChain Agent 생성
# 채팅 모델에 더 적합한 'CHAT_ZERO_SHOT_REACT_DESCRIPTION' 에이전트 사용 및 kwargs 전달
agent_final_korean = initialize_agent(
    tools,                                            # LLM이 사용가능한 Tool정보
    llm,                                              # Agent가 사용할 LLM
    agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION, # ReAct Prompt 적용
    verbose=True,                                     # ReAct 과정을 출력
    handle_parsing_errors=True,
    agent_kwargs=agent_kwargs,                        # 시스템 프롬프트 전달
    max_iterations=4,)                                # (선택): 루프 제한

/tmp/ipython-input-2160863341.py:10: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent_final_korean = initialize_agent(


## 검증 쿼리 실행  

In [13]:
# 검증 쿼리 실행
agent_final_korean.verbose = True  # 중간 과정 설정
print("--- 검증 :  단일 Tool 테스트 ---")
response = agent_final_korean.invoke("식각 공정의 상세 정보와 측정 단위를 알려줘.")
print("\n최종 답변 :")
print(response)

--- 검증 :  단일 Tool 테스트 ---


> Entering new AgentExecutor chain...
Thought: 식각 공정의 상세 정보를 알아보기 위해 Semiconductor Process DB Lookup 도구를 사용해야 합니다.
Action:
```json
{
  "action": "Semiconductor Process DB Lookup",
  "action_input": "식각"
}
```
🔍 [고급 검색] '식각' 분석 중...
[검색 성공] 식각 공정 정보 발견

Observation: 
====== 식각 공정 상세 정보 ======
- 주요 파라미터: 가스 유량, 압력(Pressure), RF 파워, 식각 시간
- 측정 단위: 가스유량(sccm), 압력(mTorr), RF파워(W), 시간(min)
- 공정 설명: 불필요한 물질을 제거하여 원하는 패턴을 만드는 공정
- 중요도: 선폭 제어와 프로파일 형성에 핵심적

Thought: 식각 공정의 상세 정보와 측정 단위를 알게 되었습니다.

Final Answer: 가스 유량(sccm), 압력(mTorr), RF 파워(W), 시간(min)


> Finished chain.

최종 답변 :
{'input': '식각 공정의 상세 정보와 측정 단위를 알려줘.', 'output': '가스 유량(sccm), 압력(mTorr), RF 파워(W), 시간(min)'}


In [14]:
# 검증 쿼리 실행
agent_final_korean.verbose = False  # 중간 과정 설정
print("--- 검증 :  단일 Tool 테스트 ---")
response = agent_final_korean.invoke("식각 공정의 상세 정보와 측정 단위를 알려줘.")
print("\n최종 답변 :")
print(response)

--- 검증 :  단일 Tool 테스트 ---
🔍 [고급 검색] '식각' 분석 중...
[검색 성공] 식각 공정 정보 발견

최종 답변 :
{'input': '식각 공정의 상세 정보와 측정 단위를 알려줘.', 'output': '가스 유량(sccm), 압력(mTorr), RF 파워(W), 시간(min)'}


In [ ]:
agent_final_korean.verbose = True   # 중간 과정 설정
print("--- 검증 :  단일 Tool 테스트 ---")
response = agent_final_korean.invoke("포토리소그래피 단계의 주요 파라미터를 알려줘.")
print("\n최종 답변 :")
print(response)

--- 검증 :  단일 Tool 테스트 ---


> Entering new AgentExecutor chain...
Thought: 포토리소그래피 단계의 주요 파라미터를 찾기 위해 Semiconductor Process DB Lookup 도구를 사용해야 합니다.
Action:
```json
{
  "action": "Semiconductor Process DB Lookup",
  "action_input": "포토리소그래피"
}
```
🔍 [고급 검색] '포토리소그래피' 분석 중...
[검색 성공] 포토리소그래피 공정 정보 발견

Observation: 
====== 포토리소그래피 공정 상세 정보 ======
- 주요 파라미터: 노광량(Exposure Dose), 초점(Focus), 감광액(PR) 두께
- 측정 단위: 노광량(mJ/cm²), 초점(μm), PR두께(nm)
- 공정 설명: 반도체 웨이퍼 위에 회로 패턴을 형성하는 핵심 공정
- 중요도: 패턴 해상도와 정밀도를 결정하는 가장 중요한 공정

Thought: 포토리소그래피 단계의 주요 파라미터가 이제 확인되었습니다.

Final Answer: 노광량, 초점, 감광액 두께


> Finished chain.

최종 답변 :
{'input': '포토리소그래피 단계의 주요 파라미터를 알려줘.', 'output': '노광량, 초점, 감광액 두께'}


In [15]:
# 검증 쿼리 실행 : 복합 질문 테스트

agent_final_korean.verbose = True  # 중간 과정 보기
print("\n\n--- 검증 : 복합 질문(Multi-Tool) 테스트 ---")
query_complex = "DUV 공정 매뉴얼에서 노광 단계 설명을 요약하고, 포토리소그래피 공정의 중요도도 알려줘."
response = agent_final_korean.invoke(query_complex)
print("\n최종 답변:")
print(response)



--- 검증 : 복합 질문(Multi-Tool) 테스트 ---


> Entering new AgentExecutor chain...
Thought: DUV 공정 매뉴얼을 읽어서 노광 단계 설명을 요약하고, 포토리소그래피 공정의 중요도도 확인해야 한다.
Action:
```json
{
  "action": "Manual Loader",
  "action_input": "DUV_manual.txt"
}
```

Observation: 
= DUV (Deep Ultraviolet) 공정 매뉴얼 =

문서 버전: 1.2
작성일: 2025-06-12

== 1. 개요 ==
DUV 리소그래피는 248nm 또는 193nm 파장의 빛을 사용하여 웨이퍼에 미세 회로 패턴을 형성하는 핵심 공정입니다.

== 2. 주요 단계 ==
2.1 웨이퍼 준비 (Wafer Preparation)
- 웨이퍼 세척 및 표면 처리

2.2 감광액 도포 (PR Coating)
- 스핀 코팅 방식을 사용하여 균일한 두께의 감광액(PR) 막을 형성합니다.

2.3 노광 (Exposure)
- 마스크에 설계된 패턴을 빛을 이용해 웨이퍼의 감광액 위로 전사시킵니다.
- **핵심 파라미터:** 노광 에너지(Dose), 초점(Focus)이 패턴의 정밀도를 결정합니다.
- 노광 장비의 정렬(Align) 정확도가 수율에 큰 영향을 미칩니다.

2.4 현상 (Development)
- 노광된 영역 또는 노광되지 않은 영역의 감광액을 선택적으로 제거하여 패턴을 완성합니다.

Thought: 노광 단계에 대한 자세한 내용이 있고, 포토리소그래피 공정의 중요도에 대한 언급도 있습니다.
Action:
```json
{
  "action": "Semiconductor Process DB Lookup",
  "action_input": "Photolithography"
}
```

🔍 [고급 검색] 'Photolithography' 분석 중...
[검색 성공] 포토리소그래피 공정 정보 발견

Observation: 


최종 답변:  
{'input': 'DUV 공정 매뉴얼에서 노광 단계 설명을 요약하고, 포토리소그래피 공정의 중요도도 알려줘.',  
'output': 'The final answer is here.\n\nDUV (Deep Ultraviolet) 리소그래피는 248nm 또는 193nm 파장의 빛을 사용하여 웨이퍼에 미세 회로 패턴을 형성하는 핵심 공정입니다.\n\n  
**노광 단계:** 마스크에 설계된 패턴을 빛을 이용해 웨이퍼의 감광액 위로 전사시킵니다. 핵심 파라미터는 노광 에너지(Dose), 초점(Focus)이며, 이들이 패턴의 정밀도를 결정합니다. 또한 노광 장비의 정렬 정확도가 수율에 큰 영향을 미칩니다.\n\n  
**포토리소그래피 공정 중요도:** 웨이퍼 위에 회로 패턴을 형성하는 가장 중요한 공정으로, 패턴 해상도와 정밀도를 결정합니다.'}



In [16]:
agent_final_korean.verbose = True
response = agent_final_korean.invoke("증착 공정의 중요도는?")
print("\n최종 답변:")
print(response)



> Entering new AgentExecutor chain...
Thought: 증착 공정에 대한 정보를 찾기 위해 Semiconductor Process DB Lookup 도구를 사용해야 합니다.
Action:
```json
{
  "action": "Semiconductor Process DB Lookup",
  "action_input": "증착"
}
```
🔍 [고급 검색] '증착' 분석 중...
[검색 성공] 증착 공정 정보 발견

Observation: 
====== 증착 공정 상세 정보 ======
- 주요 파라미터: 온도, 압력, 전구체(Precursor) 유량
- 측정 단위: 온도(°C), 압력(Torr), 유량(sccm)
- 공정 설명: 웨이퍼 위에 얇은 막을 형성하는 공정
- 중요도: 막질과 두께 균일성을 결정

Thought: 증착 공정의 중요도에 대한 정보가 있습니다.
Final Answer: 막질과 두께 균일성을 결정합니다.

> Finished chain.

최종 답변:
{'input': '증착 공정의 중요도는?', 'output': '막질과 두께 균일성을 결정합니다.'}


# HF Transformer Tool-Calling Agent 구현

In [ ]:
import os
import json
from google.colab import userdata
from langchain.agents import AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool # Tool 생성을 위한 데코레이터

## Tool로 사용할 함수 정의 (@tool 데코레이터 사용)


In [ ]:
# Tool로 사용할 함수 정의 (@tool 데코레이터 사용)
# (@tool) 데코레이터를 사용하면 함수 자체가 LangChain Tool 객체로 변환됨
# 함수의 docstring이 자동으로 tool의 description이 되어 매우 편리합니다.
@tool
def process_lookup(query: str) -> str:
    """사용자 질문에서 '포토리소그래피', '식각', '증착' 같은 공정 이름을 찾아, 해당 공정의 상세 정보(파라미터, 설명, 단위, 중요도)를 반환합니다."""
    for process_name, info in semiconductor_processes.items():
        if process_name in query:
            return json.dumps(info, ensure_ascii=False) # 결과를 JSON 문자열로 반환하여 모델이 파싱하기 쉽게 함
    return "해당 공정을 찾을 수 없습니다."

@tool
def load_manual(path: str) -> str:
    """'DUV_manual.txt'와 같이 로컬 파일 시스템에 있는 텍스트 형식의 매뉴얼을 읽어 그 내용을 문자열로 반환합니다."""
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError:
        return f"오류: '{path}' 파일을 찾을 수 없습니다."

### 도구 리스트 정의
tools = [process_lookup, load_manual]

    # 2개의 Tool로 구성된 리스트 정의 --> 단순화
    tools = [
        Tool(
            name="Semiconductor Process DB Lookup",
            func=process_lookup,
            description="반도체 공정(포토리소그래피, 식각, 증착)에 대한 상세 정보(주요 파라미터, 설명, 단위, 중요도)를 찾을 때 사용합니다. 사용자 질문에 공정 이름이 포함되어 있어야 합니다."
        ),
        Tool(
            name="Manual Loader",
            func=load_manual,
            description="로컬 텍스트 파일 형식의 매뉴얼을 읽을 때 사용합니다. 'DUV_manual.txt'와 같이 정확한 파일 경로를 입력해야 합니다."
        )
    ]

## LLM Wrapper , Agent 생성

In [ ]:
# llm = LlamaCpp(
#     model_path=model_path,
#     n_gpu_layers=40,  # GPU에 최대 레이어 오프로드 (8B 모델에 적합)
#     n_batch=512,
#     n_ctx=2048,  # 컨텍스트 길이
#     f16_kv=True,  # FP16 메모리 최적화
#     verbose=False, #True,
#     temperature=0.7,
#     max_tokens=256
# )

In [ ]:
# 프롬프트 템플릿 설정
# tool-calling을 잘 수행하도록 역할과 지시사항을 상세하게 설정
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 반도체 공정 전문가 AI 어시스턴트다. 사용자의 질문을 해결하기 위해 주어진 도구를 적극적으로 사용해야 한다. 모든 최종 답변은 반드시 상세하고 친절한 한국어로 작성해야 한다."),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"), # 에이전트의 중간 작업 내역(생각, 도구 호출 등)을 위한 자리
])

In [ ]:
# LangChain Agent 생성
# 채팅 모델에 더 적합한 'CHAT_ZERO_SHOT_REACT_DESCRIPTION' 에이전트 사용 및 kwargs 전달
agent_tool_calling = initialize_agent(
    tools,
    llm,
    agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION, # 채팅 모델에 최적화된 ReAct 에이전트
    verbose=True,
    handle_parsing_errors=True,
    agent_kwargs=agent_kwargs ) # 시스템 프롬프트 전달

# 에이전트 실행 및 검증


In [ ]:
# 에이전트 실행 및 검증
print("\n\n--- 복합 질문(Multi-Tool) 테스트 ---")
complex_query = "DUV 공정 매뉴얼에서 노광 단계 설명을 요약하고, 포토리소그래피 공정의 중요도도 알려줘."

# .invoke()를 사용하여 에이전트 실행
agent_tool_calling.verbose = False  #
response = agent_tool_calling.invoke({"input": complex_query})

print("\n\n최종 답변:")
print(response['output'])



--- 복합 질문(Multi-Tool) 테스트 ---


최종 답변:
DUV(Deep Ultraviolet) 리소그래피는 248nm 또는 193nm 파장의 빛을 사용하여 웨이퍼에 미세 회로 패턴을 형성하는 핵심 공정입니다. 주요 단계는 웨이퍼 준비, 감광액 도포, 노광, 현상으로 구성됩니다. 특히 노출 단계에서 노광 에너지(Dose)와 초점(Focus)이 패턴의 정밀도를 결정하며, 노출 장비의 정렬 정확도가 수율에 큰 영향을 미칩니다. 포토리소그래피는 반도체 소자의 집적도를 높이고 성능을 향상시키는 데 중요한 역할을 합니다.
